In [9]:
import numpy as np

class Conv3x3:
    def __init__(self, num_filters):
        self.num_filters = num_filters
        self.filters = np.random.randn(num_filters,3,3)/9

    
    def iterate_regions(self, image):
        h, w = image.shape
        for i in range(h-2):
            for j in range(w - 2):
                im_region = image[i:(i+3), j:(j+3)]
                yield im_region, i, j

    
    def forward(self, input):
        h, w = input.shape
        output = np.zeros((h-2, w-2, self.num_filters))

        for im_region, i, j in self.iterate_regions(input):
            output[i,j] = np.sum(im_region * self.filters, axis = (1,2))

        return output

In [10]:
c = Conv3x3(8)
img = c.forward(np.random.randn(4,4))
img.shape


(2, 2, 8)

In [11]:
from tensorflow.keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

In [125]:
conv = Conv3x3(8)
output = conv.forward(train_images[0])
output.shape

(26, 26, 8)

In [119]:
x[0][1][43434]

np.uint8(2)

In [13]:
class MaxPool2:

    def iterate_regions(self, image):
        h, w, _ = image.shape
        new_h, new_w = h//2, w//2

        for i in range(new_h):
            for j in range(new_w):
                im_region = image[(i*2):(i*2+2), (j*2):(j*2+2)]
                yield im_region, i, j


    def forward(self, image):
        h, w, num_filters = image.shape
        output = np.zeros((h//2, w//2, num_filters))
        for im_region, i,j in self.iterate_regions(image):
            output[i,j] = np.max(im_region, axis = (0,1))

        return output
    


In [150]:
conv = Conv3x3(8)
pool = MaxPool2()
output = conv.forward(train_images[0])
# print(type(output), output.shape)
output = pool.forward(output)
output.shape

(13, 13, 8)

In [179]:
class Softmax:
    def __init__(self, input_len, nodes):
        self.weights = np.random.randn(input_len, nodes)/input_len
        self.bias = np.zeros(nodes)


    def forward(self, input):
        input = input.flatten()
        
        input_len, nodes = self.weights.shape

        totals = np.dot(input, self.weights) +self.bias
        exp = np.exp(totals)
        return exp/np.sum(exp, axis = 0)

In [139]:
mm = np.arange(27).reshape(3,3,3)
np.amax(mm,axis= (0))

array([[18, 19, 20],
       [21, 22, 23],
       [24, 25, 26]])

In [177]:
x = np.random.randn(13, 13, 8)
x = x.flatten()
x.shape
weights = np.random.randn(x.shape[0], 10)
bias = np.random.randn(10)
total = np.dot(x, weights) +bias
total.shape, x.shape, bias.shape, weights.shape
exp = np.exp(total)
ok = exp/np.sum(exp)

In [181]:
conv = Conv3x3(8)                  # 28x28x1 -> 26x26x8
pool = MaxPool2()                  # 26x26x8 -> 13x13x8
softmax = Softmax(13*13*8, 10) # 

test_image = test_images[:1000]
test_labels = test_labels[:1000]


def forward(image, label):

    out = conv.forward((image/255)-0.5)
    out = pool.forward(out)
    out = softmax.forward(out)

    loss = -np.log(out[label])
    acc = 1 if np.argmax(out) == label else 0

    return out, loss, acc

print('MNIST CNN CLASSIFICATION')

loss = 0
num_correct = 0
for i , (im, label) in enumerate(zip(test_image, test_labels)):
    _, l, acc = forward(im, label)
    loss +=l
    num_correct += acc

    if i%100 == 99:
        print(
        '[Step %d] Past 100 steps: Average loss %.3f | Accuracy: %d%%' %(i+1, loss/100, num_correct))

        loss = 0
        num_correct = 0

MNIST CNN CLASSIFICATION
[Step 100] Past 100 steps: Average loss 2.302 | Accuracy: 8%
[Step 200] Past 100 steps: Average loss 2.302 | Accuracy: 11%
[Step 300] Past 100 steps: Average loss 2.303 | Accuracy: 9%
[Step 400] Past 100 steps: Average loss 2.302 | Accuracy: 11%
[Step 500] Past 100 steps: Average loss 2.303 | Accuracy: 11%
[Step 600] Past 100 steps: Average loss 2.303 | Accuracy: 13%
[Step 700] Past 100 steps: Average loss 2.303 | Accuracy: 10%
[Step 800] Past 100 steps: Average loss 2.303 | Accuracy: 10%
[Step 900] Past 100 steps: Average loss 2.303 | Accuracy: 8%
[Step 1000] Past 100 steps: Average loss 2.303 | Accuracy: 10%


## Backprop

1st is to backprop our way throgh the softmax and for that we need to cache some stuff from forward pass and do the backprop

In [8]:
class Softmax:
    def __init__(self, input_len, nodes):
        self.weights = np.random.randn(input_len, nodes)/input_len
        self.bias = np.zeros(nodes)


    def forward(self, input):
        self.last_input_shape = input.shape
        input = input.flatten()
        self.last_input = input
        
        input_len, nodes = self.weights.shape

        totals = np.dot(input, self.weights) +self.bias #10
        self.last_totals = totals
        exp = np.exp(totals)
        return exp/np.sum(exp, axis = 0)

    def backward(self, dL_dout, learn_rate):
        for i, gradient in enumerate(dL_dout):
            if gradient ==0:
                continue

            t_exp = np.exp(self.last_totals)
            S = np.sum(t_exp)
            dout_dt = -t_exp[i]*t_exp/S**2
            dout_dt[i] = t_exp[i]*(S-t_exp[i])/S**2

            #Gradients of totals against, w, inputs and b
            dt_dw = self.last_input
            dt_db = 1
            dt_dinputs = self.weights 

            dL_dt = gradient*dout_dt

            #Gradients of Loss against 
            dL_dw = dt_dw[np.newaxis].T @ dL_dt[np.newaxis]
            dL_db = dL_dt * dt_db
            dL_dinputs = dt_dinputs @ dL_dt
            
            self.weights -= learn_rate*dL_dw
            self.bias -= learn_rate*dL_db
            return dL_dinputs.reshape(self.last_input_shape)


In [18]:
conv = Conv3x3(8)                  # 28x28x1 -> 26x26x8
pool = MaxPool2()                  # 26x26x8 -> 13x13x8
softmax = Softmax(13*13*8, 10) # 

test_image = test_images[:1000]
test_labels = test_labels[:1000]


def forward(image, label):

    out = conv.forward((image/255)-0.5)
    out = pool.forward(out)
    out = softmax.forward(out)

    loss = -np.log(out[label])
    acc = 1 if np.argmax(out) == label else 0

    return out, loss, acc


def train(im, label, lr = 0.005):
    out, loss,acc  = forward(im, label)
    gradient = np.zeros(10)
    gradient[label] = -1/out[label]
    gradient = softmax.backward(gradient, lr)
    return loss, acc

print('MNIST CNN CLASSIFICATION')

loss = 0
num_correct = 0
for i , (im, label) in enumerate(zip(test_image, test_labels)):
    

    if i%100 == 99:
        print(
        '[Step %d] Past 100 steps: Average loss %.3f | Accuracy: %d%%' %(i+1, loss/100, num_correct))

        loss = 0
        num_correct = 0

    l, acc = train(im, label)
    loss +=l
    num_correct += acc

MNIST CNN CLASSIFICATION
[Step 100] Past 100 steps: Average loss 2.187 | Accuracy: 23%
[Step 200] Past 100 steps: Average loss 2.098 | Accuracy: 35%
[Step 300] Past 100 steps: Average loss 1.946 | Accuracy: 36%
[Step 400] Past 100 steps: Average loss 1.866 | Accuracy: 48%
[Step 500] Past 100 steps: Average loss 1.777 | Accuracy: 61%
[Step 600] Past 100 steps: Average loss 1.718 | Accuracy: 65%
[Step 700] Past 100 steps: Average loss 1.634 | Accuracy: 67%
[Step 800] Past 100 steps: Average loss 1.414 | Accuracy: 71%
[Step 900] Past 100 steps: Average loss 1.371 | Accuracy: 75%
[Step 1000] Past 100 steps: Average loss 1.306 | Accuracy: 70%
